# Assignment 06: Softmax & Multiclass Classification (100 points)

**Unit**: ML1 Supervised Learning (AI 300)  
**Topics**: Softmax function, one-hot encoding, cross-entropy loss, multiclass gradient descent, decision regions

---

## Background

For $C$ classes, **softmax regression** models $P(y=c \mid x) = \frac{\exp(w_c^T x)}{\sum_{j=1}^{C}\exp(w_j^T x)}$. Equivalently, with weight matrix $W \in \mathbb{R}^{d \times C}$ and logits $Z = XW$:

$$\text{softmax}(Z)_{ic} = \frac{\exp(Z_{ic})}{\sum_j \exp(Z_{ij})}$$

The **cross-entropy loss** is: $\mathcal{L} = -\frac{1}{n}\sum_i \sum_c Y_{ic}\log\hat{Y}_{ic}$ where $Y$ is the one-hot encoding.

### Notation

| Symbol | Shape | Description |
|--------|-------|-------------|
| $X$ | $(n, d)$ | Design matrix (with bias column) |
| $W$ | $(d, C)$ | Weight matrix |
| $Z$ | $(n, C)$ | Logits $XW$ |
| $Y$ | $(n, C)$ | One-hot encoded labels |
| $\hat{Y}$ | $(n, C)$ | Predicted probabilities (softmax output) |

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

> **WARNING !!!**
>
> - Beyond importing libraries/modules/classes/functions in the preceding cell, you are **NOT allowed to import anything else for the following purposes**:
>     - **As a part of your final solution.**
>     - **Temporarily import something to assist you to get a solution.**
>
>     **Rule of thumb:** Each part has its particular purpose to intentionally test you something. Do not attempt to find a shortcut to circumvent the rule.

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
Dataset: 3-class classification in 2D.
"""
n_per = 100
C = 3
centers = np.array([[0, 2], [-1.5, -1], [1.5, -1]])            # (3, 2)
X_raw = np.vstack([np.random.randn(n_per, 2) * 0.8 + c
                   for c in centers])                            # (300, 2)
y = np.array([0] * n_per + [1] * n_per + [2] * n_per)          # (300,)
n = len(y)
X = np.hstack([np.ones((n, 1)), X_raw])                        # (300, 3)  bias trick
d = X.shape[1]                                                   # 3

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}, classes: {np.unique(y)}")

---

## Part 1 (20 points, coding task)

**Implement numerically stable softmax and one-hot encoding.**

- **Softmax**: Use the log-sum-exp trick. Subtract the row-wise maximum before computing $\exp$: $\text{softmax}(z_i) = \exp(z_i - \max_j z_j) / \sum_j \exp(z_j - \max_j z_j)$.
- **One-hot**: Convert integer labels to a binary matrix. Must be vectorized (no loops).

*Reasoning is not required.*

In [ ]:
def softmax(Z: np.ndarray) -> np.ndarray:
    """
    Numerically stable softmax.

    Args:
        Z: (n, C) logits

    Returns:
        P: (n, C) probabilities (rows sum to 1)
    """
    ### WRITE YOUR SOLUTION HERE ###

    pass


def one_hot(y: np.ndarray, C: int) -> np.ndarray:
    """
    Convert integer labels to one-hot encoding (vectorized, no loops).

    Args:
        y: (n,) integer labels in {0, ..., C-1}
        C: number of classes

    Returns:
        Y: (n, C) one-hot matrix
    """
    ### WRITE YOUR SOLUTION HERE ###

    pass

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""
# Softmax tests
Z_test = np.array([[1, 2, 3], [1000, 1001, 1002]])               # (2, 3)
P_test = softmax(Z_test)
assert P_test.shape == (2, 3)
assert np.allclose(P_test.sum(axis=1), 1.0), "Rows must sum to 1"
assert np.all(P_test > 0), "All probabilities must be positive"
assert not np.any(np.isnan(P_test)), "NaN detected (numerical instability)"
print(f"Softmax output (large values):\n{P_test}")

# One-hot tests
y_oh = np.array([0, 2, 1, 0])
Y_oh = one_hot(y_oh, 3)
expected_oh = np.array([[1,0,0], [0,0,1], [0,1,0], [1,0,0]])
assert np.array_equal(Y_oh, expected_oh), f"One-hot incorrect:\n{Y_oh}"
print(f"One-hot:\n{Y_oh}")
print("Part 1 passed.")

""" END OF THIS PART """

---

## Part 2 (15 points, coding task)

**Implement cross-entropy loss and gradient.**

- Loss: $\mathcal{L} = -\frac{1}{n}\sum_i \sum_c Y_{ic}\log(\hat{Y}_{ic} + \epsilon)$
- Gradient: $\nabla_W \mathcal{L} = \frac{1}{n}X^T(\hat{Y} - Y)$ — shape $(d, C)$

*Reasoning is not required.*

In [ ]:
def cross_entropy_loss(
    Y_true: np.ndarray, Y_pred: np.ndarray, eps: float = 1e-12
) -> float:
    """
    Multiclass cross-entropy loss.

    Args:
        Y_true: (n, C) one-hot labels
        Y_pred: (n, C) predicted probabilities

    Returns:
        Scalar loss value
    """
    ### WRITE YOUR SOLUTION HERE ###

    pass


def softmax_gradient(
    X: np.ndarray, Y: np.ndarray, W: np.ndarray
) -> np.ndarray:
    """
    Gradient of cross-entropy loss w.r.t. W.

    Args:
        X: (n, d), Y: (n, C) one-hot, W: (d, C)

    Returns:
        grad: (d, C) gradient matrix
    """
    ### WRITE YOUR SOLUTION HERE ###

    pass

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""
Y = one_hot(y, C)                                                # (300, 3)
W_test = np.random.randn(d, C) * 0.01                           # (3, 3)

# Loss sanity
Y_pred_test = softmax(X @ W_test)                                # (300, 3)
loss_test = cross_entropy_loss(Y, Y_pred_test)
print(f"Initial loss: {loss_test:.4f} (expected ~log(3)={np.log(3):.4f})")
assert abs(loss_test - np.log(3)) < 0.5, "Initial loss should be near log(C)"

# Gradient check (numerical)
grad_analytical = softmax_gradient(X, Y, W_test)                 # (3, 3)
assert grad_analytical.shape == (d, C)
eps = 1e-5
grad_numerical = np.zeros_like(W_test)
for i in range(d):
    for j in range(C):
        Wp = W_test.copy(); Wp[i, j] += eps
        Wm = W_test.copy(); Wm[i, j] -= eps
        loss_p = cross_entropy_loss(Y, softmax(X @ Wp))
        loss_m = cross_entropy_loss(Y, softmax(X @ Wm))
        grad_numerical[i, j] = (loss_p - loss_m) / (2 * eps)
assert np.allclose(grad_analytical, grad_numerical, atol=1e-5), "Gradient mismatch"
print(f"Max gradient error: {np.max(np.abs(grad_analytical - grad_numerical)):.2e}")
print("Part 2 passed.")

""" END OF THIS PART """

---

## Part 3 (20 points, coding task)

**Implement softmax regression via gradient descent.**

Update rule: $W^{(t+1)} = W^{(t)} - \eta \cdot \nabla_W \mathcal{L}$

Initialize $W = \mathbf{0}_{d \times C}$. Record the loss at every step.

*Reasoning is not required.*

In [ ]:
def softmax_regression(
    X: np.ndarray,       # (n, d)
    Y: np.ndarray,       # (n, C) one-hot
    lr: float = 0.1,
    n_steps: int = 1000,
) -> tuple:
    """
    Train softmax regression via gradient descent.

    Args:
        X: (n, d), Y: (n, C), lr: learning rate, n_steps: iterations

    Returns:
        W: (d, C) final weight matrix
        losses: list of length n_steps
    """
    ### WRITE YOUR SOLUTION HERE ###

    pass

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""
W_trained, losses_sm = softmax_regression(X, Y, lr=0.1, n_steps=2000)
assert W_trained.shape == (d, C)
assert len(losses_sm) == 2000
assert losses_sm[-1] < losses_sm[0], "Loss should decrease"

y_pred_sm = np.argmax(X @ W_trained, axis=1)                    # (300,)
acc_sm = np.mean(y_pred_sm == y)
print(f"Accuracy: {acc_sm:.4f}")
assert acc_sm > 0.90, f"Accuracy too low: {acc_sm:.4f}"
print(f"Final loss: {losses_sm[-1]:.4f}")
print("Part 3 passed.")

""" END OF THIS PART """

---

## Part 4 (20 points, coding task)

**Visualize the training and decision regions.**

Create a 1x3 subplot grid (figure size 18x5):

1. **Left**: Loss curve (cross-entropy vs iteration).
2. **Center**: Decision regions. Create a dense grid, classify each point with `argmax`, and show colored regions with training data overlaid.
3. **Right**: Confusion matrix as a heatmap (`plt.imshow`). Implement the confusion matrix computation yourself (no sklearn).

*Reasoning is not required.*

In [ ]:
def confusion_matrix(y_true: np.ndarray, y_pred: np.ndarray, C: int) -> np.ndarray:
    """
    Compute C x C confusion matrix.
    cm[i, j] = number of samples with true label i predicted as j.

    Args:
        y_true: (n,), y_pred: (n,), C: number of classes

    Returns:
        cm: (C, C) confusion matrix
    """
    ### WRITE YOUR SOLUTION HERE ###

    pass


### WRITE YOUR SOLUTION HERE ###
# Create 1x3 subplot: loss curve, decision regions, confusion matrix

pass

""" END OF THIS PART """

---

### Softmax Reduces to Sigmoid for $C = 2$

When there are only two classes, softmax regression with weights $w_0, w_1$ is equivalent to logistic regression with $w = w_1 - w_0$: $P(y=1 \mid x) = \text{softmax}(z)_1 = \sigma((w_1 - w_0)^T x)$.

---

## Part 5 (15 points, coding task)

**Verify the softmax-sigmoid equivalence numerically.**

1. Generate a binary ($C=2$) classification dataset.
2. Train softmax regression with $C=2$.
3. Extract $w_{\text{diff}} = W_{:,1} - W_{:,0}$.
4. Show that $\sigma(X w_{\text{diff}}) \approx \text{softmax}(XW)_{:,1}$ to within $10^{-6}$.

You will need a working `sigmoid` function from Assignment 05 (re-implement here if needed).

*Reasoning is not required.*

In [ ]:
### WRITE YOUR SOLUTION HERE ###

pass

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
Verification: the difference should be negligible.
"""
# This cell will be filled by your variables from above.
# Expected: you define w_diff, prob_sigmoid, prob_softmax_1
try:
    max_diff = np.max(np.abs(prob_sigmoid - prob_softmax_1))
    print(f"Max |sigmoid - softmax[:,1]|: {max_diff:.2e}")
    assert max_diff < 1e-6, f"Equivalence not verified: max diff = {max_diff:.2e}"
    print("Part 5 passed.")
except NameError:
    print("Define prob_sigmoid and prob_softmax_1 in the cell above.")

""" END OF THIS PART """

---

## Part 6 (10 points, non-coding task)

**Analysis questions.**

1. Derive the gradient $\nabla_W \mathcal{L} = \frac{1}{n}X^T(\hat{Y} - Y)$ for softmax cross-entropy. Start from the per-sample loss and use $\frac{\partial \text{softmax}_c}{\partial z_j} = \hat{y}_c(\delta_{cj} - \hat{y}_j)$.

2. The softmax function is invariant to adding a constant: $\text{softmax}(z + c) = \text{softmax}(z)$. Explain why this means $W$ is not uniquely identifiable. How does regularization resolve this?

*Reasoning is required.*

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """